In [ ]:
#Import packages for calculation
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import pynucastro as pyna
import matplotlib.animation as animate


In [ ]:

class MainphaseStar:
    #Stellar evolution simulator that opperates on the core conceit that the star remains in the Main Phase (where the primary energy in the star comes from hydrogen fusion) and remain in thermal and hydrostatic equilibrium


    def __init__(self, temp, dens, pres, lumin, pcenthydrogen, radius):
        #Takes 4 1d-arrays of equal size and 1 float.64
        self.t = temp #in kelvin
        self.d = dens #in gram/meters**3
        self.p = pres 
        self.L = lumin
        self.pcenthydrogen = pcenthydrogen #percent by mass of hydrogen for each layer
        self.r = radius #in meters
        #obtains the number of cells as well as the size of each cell 
        self.n = len(temp)
        self.rinc = self.r / self.n
        
        #use pynucastro to load reaction rates for p-p chain and c-n-o chain reactions, assuming the rates are bounded by the slowest reaction rate 
        rl = pyna.ReacLibLibrary()
        self.ppchainrate = rl.get_rate_by_name("p(p,)d")
        self.cnochainrate = rl.get_rate_by_name("n14(p,g)o15")

        #radiation constant as defined by Stefan Bolzman Law
        self.a = 7.5657 * 10 **(-16) * 6.241509*10**12
        #units MeV (m**-3) (K**-4)

        #gravitational constant
        self.G = 6.6743*10**-13
        #units m**3 /(g*s**2)
        self.c = 299792458

    def rincrinitialize(self):
        #reinitializes the calculation for the radius increment of each cell
        self.rinc = self.r / self.n
        self.rbycell = (self.n - np.asarray(range(0, self.n)))*self.rinc

    def massbyrad(self):
        #initializes mass array for each cell, where mass is the volume * the density
        self.mass = np.zeros_like(self.t) #Mass of interior of star from each cell
        self.layermass = np.zeros_like(self.t) #mass of material in the layer composed by each cell
        for i in range(0, self.n):
            self.layermass[i] = self.d[i]*( 4/3*np.pi*((self.n-i)*self.rinc)**3 - 4/3*((self.n - i - 1)*self.rinc)**3) 
        for i in range(0, self.n):
            self.mass[i] = np.sum(self.layermass[i:])

    def fusionenergy(self, density, perhydro, temp):
        #calculate fusion energy per unit mass based on temp and density of hydrogen- using p-p and c-n-o fusion paths 
        #Note: pynucastro assumes rho density is in grams/cm**3 thus we have to convert   
        
        if perhydro == 0 or temp == 0 or density == 0:
           return 0
        #evaluates reaction rates per cm^3 
        ppeval = self.ppchainrate[0].eval(temp, rho = (density*perhydro/(10**6)))
        cnoeval = self.cnochainrate.eval(temp, rho = (density*perhydro/(10**6)))

        #convert to reaction rates per unit mass
        ppeval = ppeval*10**6/density
        cnoeval = cnoeval*10**6/density

        #evaluate energy result per unit mass then converts mega electron volt to electron volt
        energyperunitmass = (ppeval*26.72 + cnoeval*25)*1000000 
        return energyperunitmass
    
    def drdm(self, m, r, rho):   
        #mass conservation equation
        if rho == 0:
            return 0
        
        return 1/(4*np.pi*r**2 *rho)

    def dpdm(self, m, p, r):
        #hydrostatic equilibrium
        if r == 0:
            return 0
        return -self.G*m/(4*np.pi*r**4)
    
    def dLdm(self, m, L, nuc):
        #energy conservation
        return -nuc
    
    def dTdm(self, m, T, r, L, dens):
        #energy transport
        if  r == 0 or dens == 0 or m == 0:
            return 
         
        k = 10**17*dens / (T **(7/2))

        grad = ((3*k*L)/(16*np.pi*self.a*self.c*T**3))
        return -1/(4*np.pi*r**4)*grad 

        
    def massolve(self, masses):
        #Takes list of t masses to solve system at. First mass is assumed to be mass of original system 
        #returns a t by n array for each system quantity of values at each radius interval n for that mass
        #evaluates from center out for each time
        
        #initialize arrays
        self.massbyrad()
        self.rincrinitialize()
        temp = np.zeros((len(masses), self.n)) 
        dens = np.zeros((len(masses), self.n)) 
        pres = np.zeros((len(masses), self.n)) 
        radis = np.zeros((len(masses), self.n)) 
        pcent = np.zeros((len(masses), self.n)) 
        luminos = np.zeros((len(masses), self.n))
        efromrad = np.zeros((len(masses), self.n)) 
        intmass = np.zeros((len(masses), self.n))
        laymass = np.zeros((len(masses), self.n))
        temp[0, :] = self.t
        dens[0, :] = self.d
        pres[0, :] = self.p
        luminos[0, :] = self.L
        radis[0, :] = self.rbycell
        pcent[0, :] = self.pcenthydrogen
        intmass[0, :] = self.mass
        laymass[0, :] = self.layermass

        for q in range(1, len(masses)):

            #calculate the mass difference between this stage and previous mass
            massdif = masses[q] - masses[q-1]

            #evaluate per unit fusion energy at each cell and convert that into difference in mass across layers
            for i in range(0, self.n):
                efromrad[q-1, -i] = self.fusionenergy( float(dens[q-1, -i]), float(pcent[q-1, -i]), float(temp[q-1, -i]))
                if efromrad[q-1, -i] > 10**10 :
                    efromrad[q-1, -i] = 0
            pcentfusion = efromrad[q-1, :] / np.sum(efromrad[q-1, :])
            massdifbycell = massdif * pcentfusion
            laymass[q, :] = laymass[q-1, :] + massdifbycell

            for i in range(0, self.n):
                intmass[q, i] = sum(laymass[q, i:])
            #update hydrogen percentages
            pcent[q, :] = (pcent[q-1, :] * laymass[q-1, :] + massdifbycell) / laymass[q-1, :]

            #solve for radiuses
            for i in range(0, self.n):
                output = sp.integrate.solve_ivp(self.drdm, (laymass[q-1, i], laymass[q, i]), (radis[q-1, i],), args = (dens[q-1, i],))
                radis[q, i] = output["y"][-1][-1]

            #solve for preassures 
            for i in range(0, self.n):
                output = sp.integrate.solve_ivp(self.dpdm, (laymass[q-1, i], laymass[q, i]), (pres[q-1, i],), args = (radis[q, i],))
                pres[q, i] = output["y"][-1][-1]
            
            #solve for luminosities
            for i in range(0, self.n):
                output = sp.integrate.solve_ivp(self.dLdm, (laymass[q-1, i], laymass[q, i]), (luminos[q-1, i],), args = (efromrad[q-1, i],))
                luminos[q, i] = output["y"][-1][-1]
              
            #solve for temperatures
            for i in range(0, self.n):
                output = sp.integrate.solve_ivp(self.dTdm, (laymass[q-1, i], laymass[q, i]), (temp[q-1, i],), args = (radis[q, i], luminos[q, i], dens[q-1, i]))
                temp[q, i] = output["y"][-1][-1]

            for i in range(0, self.n-1):
                dens[q, i] = laymass[q, i]/(4/3*np.pi*radis[q, i]**3- 4/3*np.pi*radis[q, i+1]**3)
            dens[q, -1] = laymass[q, -1]/(4/3*np.pi*radis[q, -1]**3)
            

        return (temp, dens, pres, luminos, pcent, radis, intmass, laymass, efromrad)

In [ ]:
#1 solar mass run from MESA simulation start point
temp = np.asarray(range(0, 300))/300*(10**7.1346983)
dens = np.append(np.logspace(-6.7, 1.8, 20), np.logspace(1.8, 1.891, 280))*1000
pres =   np.asarray(range(0, 300)) * (3.4*10**11)/300*101.325*1000
radis = 700000*1000
lumin = np.logspace(-99, -0.55634, 300)
pcent = np.ones(300)*0.698965
star = MainphaseStar(temp, dens, pres, lumin, pcent, radis)

In [ ]:
star.rincrinitialize()
star.massbyrad()
star.mass
masspoints = np.linspace(1.994401955*10**33, 1.994401955*10**33*0.99999995, 800)
output = star.massolve(masspoints)
#notes: lumin, p hydro, pressure, seem to be working, but temp is wrong.

In [ ]:
def animate_circular_radial_data(
    data,
    radii=None,
    interval=100,
    cmap="plasma",
    title="Radial evolution",
    colorbar_label="Value",
    repeat=True,
    save_path=None,
    fps=10,
    figsize=(6, 6),
    grid_size=400
):
    """
    Animate radial data as a circular 2D cross-section.

    Parameters
    ----------
    data : np.ndarray
        Shape (n_radius, n_times).
        data[i, t] = value at radius-bin i and time-step t.

    radii : np.ndarray or None
        1D array of length n_radius giving the outer radius of each shell.
        If None, radii are taken to be evenly spaced from 0 to 1.

    interval : int
        Delay between frames in milliseconds.

    cmap : str
        Matplotlib colormap name.

    title : str
        Base title for the plot.

    colorbar_label : str
        Label for the colorbar.

    repeat : bool
        Whether the animation should loop.

    save_path : str or None
        If provided, saves the animation.
        Example: "star_animation.mp4" or "star_animation.gif"

    fps : int
        Frames per second for saving.

    figsize : tuple
        Figure size.

    grid_size : int
        Resolution of the square image grid.

    Returns
    -------
    anim : matplotlib.animation.FuncAnimation
    fig, ax : matplotlib figure and axes
    """

    data = np.asarray(data, dtype=float)

    if data.ndim != 2:
        raise ValueError("data must have shape (n_radius, n_times)")

    n_radius, n_times = data.shape

    if radii is None:
        radii = np.linspace(1 / n_radius, 1.0, n_radius)

    if np.any(radii <= 0):
        raise ValueError("all radii must be positive")

    if not np.all(np.diff(radii) > 0):
        raise ValueError("radii must be strictly increasing")

    r_max = radii[-1]

    x = np.linspace(-r_max, r_max, grid_size)
    y = np.linspace(-r_max, r_max, grid_size)
    X, Y = np.meshgrid(x, y)
    R = np.sqrt(X**2 + Y**2)

    outside_disk = R > r_max

    shell_idx = np.digitize(R, radii, right=True)

    shell_idx = np.clip(shell_idx, 0, n_radius - 1)

    vmin = np.nanmin(data)
    vmax = np.nanmax(data)

    fig, ax = plt.subplots(figsize=figsize)

    frame0 = data[:, 0][shell_idx]
    frame0 = np.ma.array(frame0, mask=outside_disk)

    im = ax.imshow(
        frame0,
        extent=[-r_max, r_max, -r_max, r_max],
        origin="lower",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax
    )

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(colorbar_label)

    ax.set_aspect("equal")
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.set_title(f"{title} | t = 0")

    def update(frame):
        shell_values = data[:, frame]
        image = shell_values[shell_idx]
        image = np.ma.array(image, mask=outside_disk)
        
        im.set_array(image)
        ax.set_title(f"{title} | t = {frame}")
        return [im]

    anim = animate.FuncAnimation(
        fig,
        update,
        frames=n_times,
        interval=interval,
        blit=True,
        repeat=repeat
    )

    if save_path is not None:
        if save_path.endswith(".gif"):
            anim.save(save_path, writer="pillow", fps=fps)
        else:
            anim.save(save_path, writer="ffmpeg", fps=fps)

    return anim, fig, ax

In [ ]:
temps = np.flipud(np.transpose(output[0]))
radius = np.flipud(np.transpose(output[5])[:, -1])
animate_circular_radial_data(temps, radii=radius, title="Temperature for 1 Solar Mass", colorbar_label="Temperature (K)",  save_path="Temperature1.gif")

In [ ]:
#2 solar mass run from MESA simulation start point
temp2 = np.asarray(range(0, 300))/300*(10**7.32333)
dens2 = np.append(np.logspace(-8.6, 1.3, 20), np.logspace(1.3, 1.80, 280))*1000
pres2 =   np.asarray(range(0, 300)) * (3.4*10**22)/300*101.325*1000
radis2 = 1200000*1000
lumin2 = np.logspace(-99, 0.16, 300)
pcent2 = np.ones(300)*0.697965
star2 = MainphaseStar(temp2, dens2, pres2, lumin2, pcent2, radis2)

In [ ]:
masspoints2 = np.linspace(3.78215833*10**33, 3.78215833*10**33*0.99999992, 160)
output2 = star2.massolve(masspoints2)

In [ ]:
temps2 = np.flipud(np.transpose(output[0]))
radius2 = np.flipud(np.transpose(output[5])[:, -1])
animate_circular_radial_data(temps2, radii=radius2, title="Temperature for 2 Solar Mass", colorbar_label="Temperature (K)",  save_path="Temperature2.gif")